In [24]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from datetime import datetime
import os
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from abc import ABC, abstractmethod

In [25]:
class DataParser:
    """
    Class for reading in data from the Sri Lanka surveys

    :param 
    """
 
    def __init__(
        self,
        survey_id: str,
        path_to_datasets: str,
        path_to_targets: str = None,
        path_to_hh_info: str = "../../data/processed/hh_info.csv",
        fsq24: bool = False
    ) -> None:

        # set the variables 
        self.survey_id = survey_id
        self.path_to_datasets = Path(path_to_datasets)
        self.path_to_targets = Path(path_to_targets) if path_to_targets else None
        self.hh_info = pd.read_csv(path_to_hh_info)
        self.fsq24 = fsq24
 
        self.hhid = "hhid"
        self.df = pd.DataFrame()
        self.targets = pd.DataFrame()
 
    # -------------------------
    # DATA READING
    # -------------------------
 
    def read_datasets(self, dataset_name: str) -> pd.DataFrame:
        if self.fsq24:
            path = (
                self.path_to_datasets
                / "food_security_survey_2024"
                / "data-fs-sp_final-v2.xlsx"
            )
            self.df = pd.read_excel(path)
        else:
            path = (
                self.path_to_datasets
                / "HIES_2019"
                / "HIES_2019"
                / f"{dataset_name}.csv"
            )
            self.df = pd.read_csv(path)
 
        return self.df
 
    def read_targets(self) -> pd.DataFrame:
        if self.path_to_targets is None:
            raise ValueError("No path_to_targets was provided during initialization.")
 
        self.targets = pd.read_csv(self.path_to_targets)
        self.targets = self.targets.merge(
            self.hh_info[["hhid", "adm2", "month"]],
            on="hhid",
            how="left"
        )
        return self.targets
 
    # -------------------------
    # TARGET AGGREGATION
    # -------------------------
 
    def aggregate_target(self, groups: list | None = None) -> pd.DataFrame:
        if groups is None:
            return self.targets
 
        self.targets = (
            self.targets
            .groupby(groups, as_index=False)
            .agg({
                "folate_mcg": "median",
                "fe_mg": "median",
                "zn_mg": "median",
                "vita_rae_mcg": "median",
                "overall_mar": "median"
            })
        )
        return self.targets
 
    # -------------------------
    # UTILITIES
    # -------------------------
 
    @staticmethod
    def make_hhid(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["hhid"] = (
            df["psu"].astype(str).str.strip()
            + df["snumber"].astype(str).str.strip()
            + df["hhno"].astype(str).str.strip()
        ).astype(int)
        return df
    

In [ ]:
class Feature(ABC):
    @abstractmethod
    def compute(self, df: pd.DataFrame, fsq24: bool) -> pd.DataFrame:
        """
        Compute a feature from a dataframe.
        Must return a dataframe with an ID column and the derived feature.
        """
        pass

    @abstractmethod
    def plot(self):
        pass


In [ ]:
class IncomeFeature(Feature):
    """
    Computes per-capita expenditure poverty indicator
    """
 
    def compute(self, df: pd.DataFrame, fsq24: bool) -> pd.DataFrame:
 
        df = df.copy()
 
        if fsq24:
            df["TotalConsumption"] = (
                df["TotalFexpMonthly"] + df["totalNFMonthly"]
            ) / df["HHSize"]
 
            df["asw_perexp_poor"] = (df["TotalConsumption"] < 16000).astype(int)
 
            return df[["@_id", "asw_perexp_poor"]]
 
        else:
            df = DataParser.make_hhid(df)
 
            df["asw_perexp_poor"] = (
                df["hhexppm"] / df["hhsize"] < 8104
            ).astype(int)
 
            return df[["hhid", "asw_perexp_poor"]]

In [45]:
class AnimalFeature(Feature):
    def compute(self, df: pd.DataFrame, fsq24: bool) -> pd.DataFrame:
 
        df = df.copy()

        if fsq24:
            df['asw_animal'] = (
            (df['PMT_number_animals_cattle'] >=5) |
            (df['PMT_number_animals_pig'] >=10 ) |
            (df['PMT_number_animals_goat'] >=10) |
            (df['PMT_number_animals_chicken'] >=50)
            ).astype(int)
            return df[['@_id','asw_animal']]
        else:
            df = DataParser.make_hhid(df)
            df['asw_animal'] = (
            ((df['s9_cattle_buffaloes'] == 1) & (df['cows_count'] >= 5)) |
            ((df['goats_sheeps']== 1) & (df['goat_count']>=20)) |
            ((df['pigs']==1) & (df['pigs_count']>=10)) |
            ((df['chickens']) & (df['chicken_count']>=50))

            ).astype(int)

            return df[['hhid','asw_animal']]


In [46]:
# set the paths
path_to_datasets = r"C:/Users/gabriel.battcock/OneDrive - World Food Programme/General - MIMI Project/Countries/Sri Lanka/data/"
path_to_targets = Path("../../data/processed/sl_ml_targets_2025-11-13.csv")



In [48]:
# create instances of the class to test
hies_19 = DataParser('HIES19', path_to_datasets,path_to_targets, fsq24 = False)

df = hies_19.read_datasets(dataset_name="HH_expenditure_hh_Income")
hies_19.df
hies_19.read_targets()
hies_19.aggregate_target()


FEATURE_REGISTRY = {
    "income": IncomeFeature(),
    "animal": AnimalFeature()
}


income_df = FEATURE_REGISTRY["income"].compute(df, parser.fsq24)
df = hies_19.read_datasets(dataset_name="SECTION_9_2_OWNED_LIVESTOCKS")
animal_df = FEATURE_REGISTRY["animal"].compute(df, parser.fsq24)
 
# Merge with targets if needed

targets = hies_19.read_targets()

final_df = income_df.merge(targets, on="hhid", how="left")



In [49]:

animal_df

 

,hhid,asw_animal
0,11108811,0
1,11108831,0
2,11108851,0
3,11108861,0
4,11108871,0
...,...,...
19906,19208561,0
19907,19208571,0
19908,19208581,0
19909,19208591,0


In [37]:
final_df

,hhid,asw_perexp_poor,iso3,survey,month_x,vita_rae_mcg,folate_mcg,vitb12_mcg,fe_mg,zn_mg,overall_mar,adm2,month_y
0,11108811,0,LKA,lka_hies19,1,241.031977,166.083003,1.520722,10.766685,8.347197,0.714452,11,1
1,11108831,0,LKA,lka_hies19,1,199.383626,143.233009,1.760424,10.433142,7.154734,0.671899,11,1
2,11108851,0,LKA,lka_hies19,1,202.878611,127.903376,1.592757,6.999615,5.421472,0.559565,11,1
3,11108861,0,LKA,lka_hies19,1,233.094109,156.136737,2.023155,9.046498,6.718921,0.691657,11,1
4,11108871,0,LKA,lka_hies19,1,319.620585,164.361927,1.872361,13.724426,8.869671,0.831494,11,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19906,19208561,0,LKA,lka_hies19,12,126.022109,202.143046,0.979918,12.717502,11.817528,0.680711,92,12
19907,19208571,1,LKA,lka_hies19,12,107.433567,108.421219,0.560491,8.051044,7.167088,0.455042,92,12
19908,19208581,1,LKA,lka_hies19,12,81.682584,150.781143,1.035865,9.860609,10.237976,0.589026,92,12
19909,19208591,0,LKA,lka_hies19,12,173.484320,160.906485,1.126005,9.139214,8.891278,0.633796,92,12
